# S3.2 — verifier backbone ablation (7 arms × 5 seeds × 2 LRs)

**Two code cells, and the split is deliberate.**

- **Cell 1 — preflight.** Clone, install, attach data, run the contract
  tests, run the dry run. No GPU work, no model weights, a couple of
  minutes. Safe to run interactively.
- **Cell 2 — the real run.** 70 fine-tuning runs. Hours.

They are separate because a single cell cannot be half-run: checking the
preflight output would mean starting the 70 runs and interrupting them, and
an interrupted Kaggle session still spends GPU quota.

## How to run this

1. Set **Accelerator → GPU T4**, **Internet → On**, and attach `bn_clean.csv`
   via *+ Add Input*.
2. Run **cell 1 only**. Check the three things below.
3. Then **Save Version → Save & Run All (Commit)** so the long run happens on
   Kaggle's servers rather than tied to your browser tab.

**Cell 1 must show all three, or stop:**

| Output | Expected |
|---|---|
| `git log --oneline -1` | `80dc869` or later — otherwise the push has not landed |
| pytest | `20 passed` |
| dry run | `n_train: 804`, `n_dev: 82` |

## Before you read the output

⚠️ **Read `docs/protocol.md` §"S3.2 pre-commitment" first.** The arms, the
seed count, the decision rule and the tie-break were all fixed on 2026-08-08,
before any backbone was downloaded. Reading the numbers first is how a
pre-registration quietly becomes a post-hoc story.

⚠️ **A `TIE` is a pre-registered outcome, not a failed run.** The 2025–26
Bangla literature reports three different winners on the same dataset, so a
tie is the honest and expected result. Do not re-run with different settings
to break one.

**Budget:** 70 fine-tuning runs on 804 rows of ~8-word text. Short, but not
free — expect a few hours on a T4 against a 12h session cap. If it will not
fit, split **by arm, never by seed** (splitting by seed puts one arm's seeds
in two environments and makes its SD a mixture of two things), and record the
split in the lab notebook.

## Cell 1 — preflight (minutes, no GPU)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  S3.2 preflight. Everything here is cheap and reversible.
#
#  This notebook is a RUNNER. It clones, installs, checks, and calls the
#  script. No logic lives here: anything computed in a notebook cell cannot
#  enter the paper (CLAUDE.md, working conventions).
# ══════════════════════════════════════════════════════════════════════════

%cd /kaggle/working
!rm -rf /kaggle/working/thesis
!git clone --depth 1 https://github.com/alphapie77/BSc_Thesis.git /kaggle/working/thesis
%cd /kaggle/working/thesis

# ⚠️ CHECK THIS. If it is not 80dc869 or later, the push has not landed and
#    the pre-registration this run is supposed to follow is not in the clone.
!git log --oneline -1

# ⚠️ transformers is PINNED BELOW 5. Kaggle ships 5.0.0, and setfit's own
# module chain imports transformers.training_args.default_logdir, which 5.x
# removed -- the 2026-08-08 run died on it at arm 6 of 7, after ~4 GPU-hours.
#
# The pin applies to the WHOLE run, not just the setfit arm. Running some arms
# under transformers 5 and others under 4 would put library version inside the
# comparison: Coakley et al. (2022) measured >6 pp of accuracy variation from
# hardware/software environment alone across 780 runs, and our entire
# between-arm spread is about 3 pp. The environment would be the larger effect.
!pip install -q 'transformers<5' setfit datasets pyyaml scikit-learn

import transformers, setfit
print('transformers', transformers.__version__, '| setfit', setfit.__version__)

# ── Gate 0: every arm's dependencies must import, on CPU, in ten seconds.
# ── This gate exists because its absence cost four GPU-hours.
!python -m src.verifier.s3_backbone_ablation --config configs/s3_backbone.yaml --check-arms

import shutil
from pathlib import Path

hits = sorted(Path('/kaggle/input').rglob('bn_clean.csv'))
if not hits:
    visible = [str(p) for p in Path('/kaggle/input').rglob('*') if p.is_file()][:20]
    raise FileNotFoundError(
        "bn_clean.csv is not under /kaggle/input. Attach it with '+ Add Input'. "
        f"Visible now: {visible or 'NOTHING'}"
    )
Path('data/cleaned').mkdir(parents=True, exist_ok=True)
shutil.copy(hits[0], 'data/cleaned/bn_clean.csv')
print('input:', hits[0])

# ── Gate 1: the contract tests. No GPU, no torch. If the split contract is
# ── broken, everything after this is contaminated, so stop here.
!python -m pytest tests/test_s3_backbone.py -q

# ── Gate 2: the dry run. Proves the plumbing end-to-end and re-checks that n
# ── is still 804/82 on THIS host, before any weights are downloaded. Its
# ── predictions come from a length-threshold stub and mean nothing.
!python -m src.verifier.s3_backbone_ablation --config configs/s3_backbone.yaml --dry-run

# ── The environment the real numbers will belong to. Mandatory on any
# ── non-local host (provenance fact (env)): requirements.lock.txt is
# ── Windows-frozen and does NOT describe this run.
!python -m src.common.env_snapshot --out results/env_snapshot_s3_kaggle.json

print('\nPREFLIGHT DONE. Check: commit hash, transformers 4.x, all 7 arms import, tests all passed, n_train 804 / n_dev 82.')
print('If all three are right -> Save Version -> Save & Run All (Commit).')

## Cell 2 — the real run (hours, GPU)

Do not run this interactively. Reaching it via **Save & Run All** is the
point of the split.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  The real run. 70 fine-tuning runs (7 arms x 2 LRs x 5 seeds).
#
#  The dry-run artifacts written by cell 1 are OVERWRITTEN here. That is
#  intended: a stub's output must never survive next to a real result, and
#  the `dry_run` flag inside the JSON is what tells them apart.
# ══════════════════════════════════════════════════════════════════════════

%cd /kaggle/working/thesis

!python -m src.verifier.s3_backbone_ablation --config configs/s3_backbone.yaml

# ── Package for download, so results are committed WITH their notebook entry.
import json
import zipfile
from pathlib import Path

res = json.loads(Path('results/s3_backbone_ablation.json').read_text(encoding='utf-8'))
assert res['result']['dry_run'] is False, (
    'the result file is still from the dry run -- the real run did not write it'
)

OUT = Path('/kaggle/working/s3_backbone_outputs.zip')
wanted = [
    'results/s3_backbone_ablation.md',
    'results/s3_backbone_ablation.json',
    'results/s3_backbone_per_seed.csv',
    'results/env_snapshot_s3_kaggle.json',
]
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in wanted:
        if Path(f).exists():
            z.write(f)
        else:
            print('MISSING:', f)
print('wrote', OUT)

print(open('results/s3_backbone_ablation.md', encoding='utf-8').read())